# Titanic
## Score: .79425


In [57]:
import numpy as np
import pandas as pd
from pathlib import Path
from catboost import CatBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 42
np.random.seed(SEED)
ROOT = Path.cwd()
DATA = ROOT / "titanic"
RARE = {"Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major", "Rev", "Sir", "Jonkheer", "Dona"}

In [58]:
def build_xy(train_df: pd.DataFrame, test_df: pd.DataFrame):
    y = train_df["Survived"].to_numpy()
    tr = train_df.drop(columns=["Survived"]).copy()
    te = test_df.copy()

    ticket_vc = tr["Ticket"].value_counts()
    med_fare = tr["Fare"].median()
    mode_emb = tr["Embarked"].mode().iloc[0]
    tr["Fare"] = tr["Fare"].fillna(med_fare)
    te["Fare"] = te["Fare"].fillna(med_fare)
    tr["Embarked"] = tr["Embarked"].fillna(mode_emb)
    te["Embarked"] = te["Embarked"].fillna(mode_emb)

    def impute_age(ref: pd.DataFrame, df: pd.DataFrame) -> None:
        gmed = ref["Age"].median()
        for (sex, pcl), m in ref.groupby(["Sex", "Pclass"])["Age"].median().items():
            mm = m if pd.notna(m) else gmed
            msk = (df["Sex"] == sex) & (df["Pclass"] == pcl) & df["Age"].isna()
            df.loc[msk, "Age"] = mm
        df["Age"] = df["Age"].fillna(gmed)

    impute_age(tr, tr)
    impute_age(tr, te)

    def feats(df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        out["Title"] = out["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
        out["Title"] = out["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
        out["Title"] = out["Title"].where(~out["Title"].isin(RARE), "Rare")
        out["FamilySize"] = out["SibSp"] + out["Parch"] + 1
        out["IsAlone"] = (out["FamilySize"] == 1).astype(int)
        out["Deck"] = out["Cabin"].apply(lambda x: str(x)[0] if pd.notna(x) and str(x) and str(x)[0].isalpha() else "U")
        out["TicketGroupSize"] = out["Ticket"].map(ticket_vc).fillna(1).astype(int)
        out["FarePerPerson"] = out["Fare"] / out["FamilySize"].clip(lower=1)
        out["HasCabin"] = out["Cabin"].notna().astype(int)
        out["AgeBin"] = pd.cut(out["Age"], bins=[0, 12, 18, 35, 60, 200], labels=["c", "t", "y", "a", "s"]).astype(str)
        out["Surname"] = out["Name"].str.split(",").str[0].str.strip()
        out = out.drop(columns=["Name", "Ticket", "Cabin", "PassengerId"], errors="ignore")
        out["Pclass"] = out["Pclass"].astype(str)
        for c in ["Sex", "Embarked", "Title", "Deck", "AgeBin", "Surname"]:
            out[c] = out[c].astype(str)
        return out

    X = feats(tr)
    X_test = feats(te)
    return X, y, X_test


def add_oof_target_encoding(X: pd.DataFrame, y: np.ndarray, X_test: pd.DataFrame, cols, n_splits=5, alpha=20.0):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    y_s = pd.Series(y)
    global_mean = y_s.mean()
    X_new = X.copy()
    X_test_new = X_test.copy()

    for col in cols:
        oof_col = np.zeros(len(X), dtype=float)
        for tr_i, va_i in cv.split(X, y):
            tr_c = X.iloc[tr_i][col]
            y_tr = y_s.iloc[tr_i]
            stat = y_tr.groupby(tr_c).agg(["mean", "count"])
            smooth = (stat["mean"] * stat["count"] + global_mean * alpha) / (stat["count"] + alpha)
            oof_col[va_i] = X.iloc[va_i][col].map(smooth).fillna(global_mean).to_numpy()
        full_stat = y_s.groupby(X[col]).agg(["mean", "count"])
        full_smooth = (full_stat["mean"] * full_stat["count"] + global_mean * alpha) / (full_stat["count"] + alpha)
        X_new[col + "_te"] = oof_col
        X_test_new[col + "_te"] = X_test[col].map(full_smooth).fillna(global_mean).to_numpy()
    return X_new, X_test_new


train_raw = pd.read_csv(DATA / "train.csv")
test_raw = pd.read_csv(DATA / "test.csv")
pid = test_raw["PassengerId"].values
X, y, X_test = build_xy(train_raw, test_raw)
cat_cols = ["Sex", "Embarked", "Title", "Deck", "Pclass", "AgeBin", "Surname"]
te_cols = ["Title", "Deck", "Embarked", "Pclass", "AgeBin", "Surname"]
X_te, X_test_te = add_oof_target_encoding(X, y, X_test, te_cols, n_splits=5, alpha=18.0)

hgb_features = [
    "Age", "SibSp", "Parch", "Fare", "FamilySize", "IsAlone", "TicketGroupSize", "FarePerPerson", "HasCabin"
] + [c + "_te" for c in te_cols]
X_hgb = X_te[hgb_features].to_numpy()
X_test_hgb = X_test_te[hgb_features].to_numpy()

In [59]:
Xc_tr, Xc_va, y_tr, y_va, idx_tr, idx_va = train_test_split(
    X,
    y,
    np.arange(len(y)),
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)
Xh_tr = X_hgb[idx_tr]
Xh_va = X_hgb[idx_va]

cb = CatBoostClassifier(
    iterations=7000,
    learning_rate=0.025,
    depth=6,
    l2_leaf_reg=3.0,
    random_seed=SEED,
    thread_count=1,
    verbose=False,
)
cb.fit(
    Xc_tr,
    y_tr,
    cat_features=cat_cols,
    eval_set=(Xc_va, y_va),
    early_stopping_rounds=150,
    verbose=False,
)

hgb = HistGradientBoostingClassifier(
    learning_rate=0.03,
    max_iter=700,
    max_depth=7,
    min_samples_leaf=8,
    l2_regularization=0.25,
    random_state=SEED,
)
hgb.fit(Xh_tr, y_tr)

p_cb = cb.predict_proba(Xc_va)[:, 1]
p_hgb = hgb.predict_proba(Xh_va)[:, 1]
blend_va = 0.68 * p_cb + 0.32 * p_hgb
pred_va = (blend_va >= 0.5).astype(int)
print("holdout_acc", round(accuracy_score(y_va, pred_va), 5))
print(confusion_matrix(y_va, pred_va))

holdout_acc 0.82123
[[97 13]
 [19 50]]


In [60]:
best_iter = cb.get_best_iteration()
if best_iter is None:
    best_iter = cb.tree_count_

cb_full = CatBoostClassifier(
    iterations=max(50, best_iter + 1),
    learning_rate=0.025,
    depth=6,
    l2_leaf_reg=3.0,
    random_seed=SEED,
    thread_count=1,
    verbose=False,
)
cb_full.fit(X, y, cat_features=cat_cols, verbose=False)

hgb_full = HistGradientBoostingClassifier(
    learning_rate=0.03,
    max_iter=700,
    max_depth=7,
    min_samples_leaf=8,
    l2_regularization=0.25,
    random_state=SEED,
)
hgb_full.fit(X_hgb, y)

p_cb_t = cb_full.predict_proba(X_test)[:, 1]
p_hgb_t = hgb_full.predict_proba(X_test_hgb)[:, 1]
blend_t = 0.68 * p_cb_t + 0.32 * p_hgb_t
submission = pd.DataFrame({"PassengerId": pid, "Survived": (blend_t >= 0.5).astype(int)})
submission.to_csv(ROOT / "submission.csv", index=False)
print(ROOT / "submission.csv")
submission.head(10)

c:\Users\ol1v3_7dwns5u\OneDrive\Desktop\ACTIVE\Titanic\submission.csv


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
5,897,0
6,898,1
7,899,0
8,900,1
9,901,0
